# Minimal LoRA XLM-R Classifier + Bayes MAE + Ordinal Contrastive

Small `xlm-roberta-base` notebook: train a 5-class classifier with LoRA, decode posteriors with the Bayes action for MAE, and add a simple in-batch ordinal soft contrastive loss on CLS embeddings.

For each anchor `i`, the contrastive target over other batch items is `q_ij ∝ exp(-TAU_Y * |y_i - y_j|)`. The model distribution is the softmax of embedding similarities, `p_ij ∝ exp(z_i^T z_j / CONTRASTIVE_T)`. The added loss is `-sum_i sum_{j != i} q_ij log p_ij`.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import inspect
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, Value
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_classifier_bayes_mae_ordcon_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

EPOCHS = 1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1024
LR = 1.5e-4
FP16 = torch.cuda.is_available()

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

# Ordinal contrastive knobs.
LAMBDA_ORDCON = 0.10
TAU_Y = 1.25
CONTRASTIVE_T = 0.10

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())
train_df.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def tokenize(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [int(x) for x in batch["label"]]
    return out


def to_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Value("int64"))
    ds.set_format("torch")
    return ds


train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)

## Model

In [ ]:
def ordinal_soft_contrastive_loss(embeddings, labels, tau_y=TAU_Y, temperature=CONTRASTIVE_T):
    if embeddings.shape[0] < 2:
        return embeddings.new_zeros(())

    z = F.normalize(embeddings, p=2, dim=-1)
    logits = z @ z.T / temperature
    labels = labels.float().view(-1)
    mask = ~torch.eye(labels.shape[0], dtype=torch.bool, device=labels.device)

    distances = torch.abs(labels[:, None] - labels[None, :])
    target_logits = -tau_y * distances
    target_logits = target_logits.masked_fill(~mask, -torch.inf)
    q = F.softmax(target_logits, dim=1).detach()

    logits = logits.masked_fill(~mask, -torch.inf)
    log_p = F.log_softmax(logits, dim=1)
    return -(q * log_p).masked_fill(~mask, 0.0).sum(dim=1).mean()


class XLMRClassifierOrdCon(nn.Module):
    def __init__(self, model_id=MODEL_ID, n_classes=N_CLASSES):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_id)
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
            lora_dropout=LORA_DROPOUT,
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.backbone = get_peft_model(self.backbone, lora_config)

        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.dropout(outputs.last_hidden_state[:, 0, :])
        logits = self.classifier(embeddings)

        loss = None
        if labels is not None:
            labels = labels.long().view(-1)
            loss_cls = F.cross_entropy(logits, labels)
            loss_ordcon = ordinal_soft_contrastive_loss(embeddings, labels)
            loss = loss_cls + LAMBDA_ORDCON * loss_ordcon
        return {"loss": loss, "logits": logits}


def make_model():
    model = XLMRClassifierOrdCon()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model

## Metrics

In [ ]:
def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(pred - classes), axis=1) for pred in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    map_preds = probs.argmax(axis=1).astype(int)
    bayes_preds = bayes_mae_decode(probs)
    expected_score = probs @ np.arange(N_CLASSES)
    return {
        "accuracy": float(accuracy_score(labels, map_preds)),
        "map_mae": float(mean_absolute_error(labels, map_preds)),
        "bayes_mae": float(mean_absolute_error(labels, bayes_preds)),
        "expected_score_mae": float(mean_absolute_error(labels, expected_score)),
    }


def make_training_args(**kwargs):
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)

## Train

In [ ]:
args = make_training_args(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    #overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=EPOCHS,
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=FP16,
    report_to=[],
    remove_unused_columns=False,
    seed=SEED,
)

model = make_model()
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
metrics

## Validation decoders

In [ ]:
val_out = trainer.predict(val_ds)
val_probs = softmax_np(val_out.predictions)
val_labels = val_out.label_ids.astype(int)
map_preds = val_probs.argmax(axis=1).astype(int)
bayes_preds = bayes_mae_decode(val_probs)

summary = pd.DataFrame([
    {"decoder": "classifier_map", "mae": mean_absolute_error(val_labels, map_preds), "counts": np.bincount(map_preds, minlength=N_CLASSES).tolist()},
    {"decoder": "classifier_bayes_mae", "mae": mean_absolute_error(val_labels, bayes_preds), "counts": np.bincount(bayes_preds, minlength=N_CLASSES).tolist()},
])
display(summary.sort_values("mae"))

val_predictions = pd.concat(
    [
        val_df.reset_index(drop=True),
        pd.Series(map_preds, name="map_pred"),
        pd.Series(bayes_preds, name="bayes_mae_pred"),
        pd.DataFrame(val_probs, columns=[f"p_{k}" for k in range(N_CLASSES)]),
    ],
    axis=1,
)
val_predictions.head()

## Save adapter and validation predictions

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR / "adapter"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "tokenizer"))
val_predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
summary.to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)
print(OUTPUT_DIR)

## Optional test submission

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    test_logits = trainer.predict(test_ds).predictions
    test_probs = softmax_np(test_logits)
    test_preds = bayes_mae_decode(test_probs)

    submission_path = OUTPUT_DIR / "submission_classifier_bayes_mae_ordcon.csv"
    pd.DataFrame({"id": test_df["id"], "label": test_preds.astype(int)}).to_csv(submission_path, index=False)
    print("counts:", np.bincount(test_preds, minlength=N_CLASSES).tolist())
    print(submission_path)
else:
    print("No test CSV found:", TEST_CSV)